In [ ]:
# !pip install scikit-learn-extra
# !pip install "numpy<2.0"
# !pip install arch
import os
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"

In [ ]:
import numpy as np
from statsmodels.tsa.arima_process import ArmaProcess

def durbin_levinson(pacf):
    p = len(pacf)
    phi = np.zeros(p)
    phis = np.zeros((p,p))
    phi[0] = pacf[0]
    phis[0,0] = pacf[0]
    for k in range(1, p):
        sum_val = pacf[k]
        for j in range(k):
            sum_val -= phis[k-1, j] * pacf[k-j-1]
        phi[k] = sum_val
        for j in range(k):
            phis[k,j] = phis[k-1,j] - pacf[k] * phis[k-1,k-j-1]
        phis[k,k] = pacf[k]
    return phis[-1]

def sample_crp_with_min_clusters(alpha, n):
    """Generate cluster assignments for n items from a CRP with concentration alpha,
    conditioned on having more than 1 cluster."""
    while True:
        assignments = []
        cluster_counts = []
        for i in range(n):
            if i == 0:
                assignments.append(0)
                cluster_counts.append(1)
            else:
                probs = np.array(cluster_counts + [alpha])
                probs = probs / probs.sum()
                choice = np.random.choice(len(probs), p=probs)
                assignments.append(choice)
                if choice == len(cluster_counts):
                    cluster_counts.append(1)
                else:
                    cluster_counts[choice] += 1
        
        assignments = np.array(assignments)
        if len(np.unique(assignments)) > 1:
            return assignments
        # else repeat the simulation until more than one cluster is produced

def compute_qaf(ts, tau_1 = 0.5, tau_2 = 0.5, lag = 1):
    ts = np.asarray(ts)
    l_ts = len(ts)

    ts_1 = ts[:l_ts - lag]
    ts_2 = ts[lag:]

    q_tau_1 = np.quantile(ts, tau_1)
    q_tau_2 = np.quantile(ts, tau_2)

    indicators_1 = (ts_1 <= q_tau_1).astype(float)
    indicators_2 = (ts_2 <= q_tau_2).astype(float)

    numerator = (1/l_ts) * np.sum(indicators_1 * indicators_2) - tau_1 * tau_2
    denominator = np.sqrt(tau_1 * (1 - tau_1) * tau_2 * (1 - tau_2))

    return numerator/denominator

def compute_qafs(ts, levels, lags):
    levels = np.asarray(levels)
    lags = np.asarray(lags)

    l_levels = len(levels)
    l_lags = len(lags)
    array_features = np.zeros((l_levels, l_levels, l_lags))

    for i in range(l_levels):
        for j in range(l_levels):
            for k in range(l_lags):
                array_features[i, j, k] = compute_qaf(
                    ts,
                    tau_1=levels[i],
                    tau_2=levels[j],
                    lag=lags[k]
                )

    return array_features.ravel(order="F")

def simulate_ar3_clustering_with_qaf(series_length, random_state=None):
    rng = np.random.default_rng(random_state)
    
    # Sample number of time series n from discrete uniform [10, 200]
    n = rng.integers(10, 201)
    
    # Sample CRP concentration parameter alpha from Exp(1)
    alpha1 = rng.exponential(1.0)
    
    # Generate cluster assignments using CRP with concentration alpha1
    cluster_assignments = sample_crp_with_min_clusters(alpha1, n)
    
    # Identify distinct clusters and number of clusters K
    unique_clusters = np.unique(cluster_assignments)
    K = len(unique_clusters)
    
    # For each cluster, sample PACF and noise variance, convert to AR coefficients
    cluster_ar_coefs = []
    cluster_noise_vars = []
    for _ in range(K):
        pacf = rng.uniform(-1, 1, size=3)
        ar_coefs = durbin_levinson(pacf)
        ar_coefs_full = np.concatenate(([1], -ar_coefs))
        cluster_ar_coefs.append(ar_coefs_full)
        noise_var = rng.uniform(0.1, 2)
        cluster_noise_vars.append(noise_var)
    
    # Construct feature matrix by simulating time series and calculating QAF features
    features = []
    for i in range(n):
        # Map cluster label to index used in parameters list
        cluster_idx = np.where(unique_clusters == cluster_assignments[i])[0][0]
        arma_process = ArmaProcess(cluster_ar_coefs[cluster_idx], [1])
        ts = arma_process.generate_sample(
            nsample=series_length,
            scale=np.sqrt(cluster_noise_vars[cluster_idx]),
            burnin=100
        )
        
        # QAF features instead of autocorrelations
        feat_vec = compute_qafs(ts, levels=(0.1, 0.5, 0.9), lags=(1, 2, 3))
        features.append(feat_vec)

    D = np.array(features)                      # feature matrix (n, dim_qaf)
    C = np.array(cluster_assignments)          # cluster labels (n,)
    S = (C[:, None] == C[None, :]).astype(int) # similarity matrix (n, n)
    
    return D, C, S

# Example usage
D, C, S = simulate_ar3_clustering_with_qaf(series_length=100, random_state=42)
print("Number of series (n):", D.shape[0])
print("Number of features:", D.shape[1])
print("Clusters assigned:", len(np.unique(C)))
print("Similarity matrix shape:", S.shape)

In [ ]:
rng = np.random.default_rng()
alpha = rng.uniform(0.01, 0.30)
beta = rng.uniform(0.70, 1 - alpha)
print(alpha)
print(beta)
is_strictly_stationary(alpha, beta, 3, num_samples=50000)

sample_crp_with_min_clusters(rng.exponential(0.5), 100)

In [ ]:
ts = np.array([1, 3, 4, 5, 3, 2, 1, 3, 1, 2, 4, 5, 6, 1, 2, 3, 4, 1, 1, 2])
levels = np.array([0.1, 0.5, 0.9])
lags = np.array([1, 2, 3])

# Compute QAFs
result_2 = compute_qafs(ts, levels = (0.1, 0.5, 0.9), lags = (1, 2, 3))
print(result_2)

In [ ]:
import torch

def prepare_pairwise_data(D, S):
    ''' Returns pairs (concat) and targets for all i < j '''
    n = D.shape[0]
    pairs = []
    targets = []
    for i in range(n):
        for j in range(i+1, n):
            pair = np.concatenate([D[i], D[j]])  
            target = S[i, j]                     # 1 if same cluster, 0 if not
            pairs.append(pair)
            targets.append(target)
    return np.array(pairs), np.array(targets)

# Example preparation:
pairs, targets = prepare_pairwise_data(D, S)
print("Pairs shape:", pairs.shape)       
print("Targets shape:", targets.shape)   
print(pairs)
print(targets)

In [ ]:
# Constructing the network

import torch
import torch.nn as nn

class PairwiseNN(nn.Module):
    def __init__(self, vector_dim=27, embed_dim=128, hidden_dim=256):
        super().__init__()
        # Embedding network phi applied independently to each vector
        self.phi = nn.Sequential(
            nn.Linear(vector_dim, embed_dim),
            nn.ReLU(),
            nn.Linear(embed_dim, embed_dim),
            nn.ReLU()
        )
        # Prediction network rho that takes aggregated embedding
        self.rho = nn.Sequential(
            nn.Linear(embed_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
      
        v1 = x[:, :27]  # first vector
        v2 = x[:, 27:]  # second vector

        emb1 = self.phi(v1)  # embed first vector
        emb2 = self.phi(v2)  # embed second vector

        combined = emb1 + emb2  # symmetric aggregation (sum)

        out = self.rho(combined)  # predict
        return out.squeeze(-1)  # shape: (batch_size,)


In [ ]:
# Training

import torch
import torch.nn as nn
import numpy as np

series_lengths = [300]
num_epochs = 1
units_per_epoch = 500000  # total simulated datasets per epoch
datasets_per_batch = 64   # datasets per batch update
early_stopping_patience = 100
models_by_length = {}

for T in series_lengths:
    print(f"\nTraining model for series length: {T}")
    model = PairwiseNN(vector_dim=27, embed_dim=128, hidden_dim=256)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.BCELoss()
    best_val_loss = float('inf')
    patience_counter = 0

    for epoch in range(num_epochs):
        losses = []
        indices = np.arange(units_per_epoch // datasets_per_batch)
        for _ in indices:

            batch_num = len(losses) + 1
            print(f"Batch {batch_num}")
            
            batch_pairs = []
            batch_targets = []
            for _ in range(datasets_per_batch):
        
                D, C, S = simulate_ar3_clustering_with_qaf(series_length=T)
                pairs, targets = prepare_pairwise_data(D, S)
                batch_pairs.append(pairs)
                batch_targets.append(targets)
            
            pairs_tensor = torch.tensor(np.vstack(batch_pairs)).float()
            targets_tensor = torch.tensor(np.hstack(batch_targets)).float()
            model.train()
            optimizer.zero_grad()
            outputs = model(pairs_tensor)
            loss = criterion(outputs, targets_tensor)
            loss.backward()
            optimizer.step()
            losses.append(loss.item())

        mean_loss = np.mean(losses)

        # Validation step
        
        model.eval()
        Dv, Cv, Sv = simulate_ar3_clustering_with_qaf(series_length=T)
        pairs_val, targets_val = prepare_pairwise_data(Dv, Sv)
        pairs_tensor_val = torch.tensor(pairs_val).float()
        targets_tensor_val = torch.tensor(targets_val).float()
        with torch.no_grad():
            outputs_val = model(pairs_tensor_val)
            val_loss = criterion(outputs_val, targets_tensor_val).item()

        print(f"Epoch {epoch+1}, train loss: {mean_loss:.4f}, val loss: {val_loss:.4f}")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0
            
        else:
            patience_counter += 1

        if patience_counter >= early_stopping_patience:
            print(f"Early stopping at epoch {epoch+1} (best val loss {best_val_loss:.4f})")
            break

    models_by_length[f"model_{T}"] = model

print("\nTraining completed. Models saved in memory.")

torch.save(model.state_dict(), f"output/model_{T}.pth")

In [ ]:
import numpy as np

def simulate_setar1(n, phi1, phi2, r, sigma=1.0, burnin=100):
    
    T = n + burnin
    y = np.zeros(T)

    # White noise
    eps = np.random.normal(0, sigma, size=T)

    # Iterate
    for t in range(1, T):
        if y[t - 1] <= r:
            y[t] = phi1 * y[t - 1] + eps[t]
        else:
            y[t] = phi2 * y[t - 1] + eps[t]

    return y[burnin:]

def simulate_setar1_with_qaf(series_length, random_state=None):
    rng = np.random.default_rng(random_state)

    n = rng.integers(10, 201)
    alpha1 = rng.exponential(1.0)
    cluster_assignments = sample_crp_with_min_clusters(alpha1, n)

    unique_clusters = np.unique(cluster_assignments)
    K = len(unique_clusters)

    cluster_setar_params = []
    for _ in range(K):
        phi_1 = rng.uniform(-1, 1)
        phi_2 = rng.uniform(-1, 1)
        r = rng.uniform(-0.75, 0.75)
        cluster_setar_params.append((phi_1, phi_2, r))

    features = []

    for i in range(n):
        cluster_idx = np.where(unique_clusters == cluster_assignments[i])[0][0]
        phi_1, phi_2, r = cluster_setar_params[cluster_idx]

        ts = simulate_setar1(series_length, phi_1, phi_2, r)

        feat_vec = compute_qafs(ts, levels=(0.1, 0.5, 0.9), lags=(1, 2, 3))
        features.append(feat_vec)

    D = np.array(features)
    C = np.array(cluster_assignments)
    S = (C[:, None] == C[None, :]).astype(int)

    return D, C, S



In [ ]:
# Evaluation

import torch
import numpy as np
import networkx as nx
import community.community_louvain as community_louvain
import pandas as pd
from sklearn.cluster import SpectralClustering, KMeans, AgglomerativeClustering
from sklearn_extra.cluster import KMedoids
from sklearn.metrics import adjusted_rand_score

num_trials = 10000
series_lengths = [300]

results = {T: {
    "spectral": [],
    "kmeans": [],
    "kmedoids": [],
    "agglo": [],
    "louvain_pred": []
} for T in series_lengths}

for T in series_lengths:
    model = models_by_length[f"model_{T}"]
    print(f"\nEvaluating model trained on series length {T}")
    for trial in range(num_trials):
        
        if np.random.rand() < 0.5:
            D_test, C_test, S_test = simulate_setar1_with_qaf(series_length=300)
        else:
            D_test, C_test, S_test = simulate_ar3_clustering_with_qaf(series_length=300)
        
        n_series = D_test.shape[0]
        K_true = len(np.unique(C_test))  # true number of clusters per dataset

        pairs, targets = prepare_pairwise_data(D_test, S_test)
        pairs_tensor = torch.tensor(pairs).float()
        
        with torch.no_grad():
            pred_sim_probs = model(pairs_tensor).detach().cpu().numpy()
            pred_sim_matrix = np.zeros((n_series, n_series), dtype=float)
            idx = 0
            for i in range(n_series):
                for j in range(i + 1, n_series):
                    pred_sim_matrix[i, j] = pred_sim_probs[idx]
                    pred_sim_matrix[j, i] = pred_sim_probs[idx]
                    idx += 1
            np.fill_diagonal(pred_sim_matrix, 1.0)

            spectral = SpectralClustering(n_clusters=K_true, affinity="precomputed")
            spectral_labels = spectral.fit_predict(pred_sim_matrix)
            spectral_ari = adjusted_rand_score(C_test, spectral_labels)
            results[T]["spectral"].append(spectral_ari)

            kmeans = KMeans(n_clusters=K_true, n_init=500)
            kmeans_labels = kmeans.fit_predict(D_test)
            kmeans_ari = adjusted_rand_score(C_test, kmeans_labels)
            results[T]["kmeans"].append(kmeans_ari)

            kmedoids = KMedoids(n_clusters=K_true, init="k-medoids++")
            kmedoids_labels = kmedoids.fit_predict(D_test)
            kmedoids_ari = adjusted_rand_score(C_test, kmedoids_labels)
            results[T]["kmedoids"].append(kmedoids_ari)

            agglo = AgglomerativeClustering(n_clusters=K_true, linkage="ward")
            agglo_labels = agglo.fit_predict(D_test)
            agglo_ari = adjusted_rand_score(C_test, agglo_labels)
            results[T]["agglo"].append(agglo_ari)

            G_pred = nx.from_numpy_array(pred_sim_matrix)
            partition_pred = community_louvain.best_partition(G_pred)
            louvain_pred_labels = np.array([partition_pred[i] for i in range(n_series)])
            louvain_pred_ari = adjusted_rand_score(C_test, louvain_pred_labels)
            results[T]["louvain_pred"].append(louvain_pred_ari)

        if (trial + 1) % 100 == 0:
            print(f"Trial {trial + 1}/{num_trials} completed for series length {T}")

# Prepare summary table
summary_data = {
    "Series Length": [],
    "Spectral Clustering": [],
    "Louvain on Affinities": [],
    "KMeans on Features": [],
    "KMedoids on Features": [],
    "Agglomerative (Ward)": [],
}
for T in series_lengths:
    summary_data["Series Length"].append(T)
    summary_data["Spectral Clustering"].append((results[T]["spectral"]))
    summary_data["Louvain on Affinities"].append((results[T]["louvain_pred"]))
    summary_data["KMeans on Features"].append((results[T]["kmeans"]))
    summary_data["KMedoids on Features"].append((results[T]["kmedoids"]))
    summary_data["Agglomerative (Ward)"].append((results[T]["agglo"]))

df_summary = pd.DataFrame(summary_data)
import os
output_dir = "df_summary_scenario_4"
os.makedirs(output_dir, exist_ok=True)
df_summary.to_csv(f'{output_dir}/df_summary_scenario_4.csv', index=False)
# print(df_summary.to_string(index=False, float_format='%.3f'))

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import ast
import numpy as np

csv_path = 'df_summary_scenario_4/df_summary_scenario_4.csv'
df_summary = pd.read_csv(csv_path)

# Since you only have one fixed series length, take the first row
row = df_summary.iloc[0]

data = [
    ast.literal_eval(row["Spectral Clustering"]),
    ast.literal_eval(row["Louvain on Affinities"]),
    ast.literal_eval(row["KMeans on Features"]),
    ast.literal_eval(row["KMedoids on Features"]),
    ast.literal_eval(row["Agglomerative (Ward)"])
]

labels = [
    "Proposed (spectral)",
    "Proposed (Louvain)",
    "K-Means",
    "K-Medoids",
    "Hierarchical"
]

plt.figure(figsize=(10, 6))
box = plt.boxplot(data, tick_labels=labels, patch_artist=True)

colors = ['blue', 'orange', 'green', 'red', 'purple']
for patch, color in zip(box['boxes'], colors):
    patch.set_facecolor(color)

plt.ylabel('ARI', fontsize=16)
plt.title(f'Scenario 4 (T=300)', fontsize=18)
plt.xticks(fontsize=14, rotation=20)
plt.yticks(fontsize=14)
plt.grid(True, axis='y')

medians = [np.median(x) for x in data]
print("Medians:")
for label, med in zip(labels, medians):
    print(f"{label}: {med:.4f}")

plt.savefig('fig_boxplot.png', dpi=300, bbox_inches='tight')
plt.show()